In [0]:
from pyspark.sql.functions import col,round

matches = spark.table("gold_dev.default.fact_match_results")
elo = spark.table("gold_dev.default.fact_elo")

#get home and away team elo 
homeElo = elo.select("MatchKey", col("TeamId").alias("HomeTeamId"), col("EloBefore").alias("HomeTeamElo"))
awayElo = elo.select("MatchKey", col("TeamId").alias("AwayTeamId"), col("EloBefore").alias("AwayTeamElo"))

#creating training dataframe
training_df = (
    matches
    .join(homeElo, ["MatchKey", "HomeTeamId"], "inner")
    .join(awayElo, ["MatchKey", "AwayTeamId"], "inner")
    .withColumn("EloDiff", round(col("HomeTeamElo") - col("AwayTeamElo")))
    .select("HomeTeamElo", "AwayTeamElo", "EloDiff", "IsNeutral", "HomeWinFlag")
    .dropna()
)


In [0]:
display(training_df)

In [0]:
#convert to pandas df

df = training_df.toPandas()

#split data
X = df[["HomeTeamElo", "AwayTeamElo", "EloDiff", "IsNeutral"]]
y = df["HomeWinFlag"]

In [0]:
#create the test and split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle = False, random_state=42)

In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#train the model
model = LogisticRegression(solver = "lbfgs", max_iter = 1000)
model.fit(X_train_scaled, y_train)

In [0]:
#evaluation metrics
from sklearn.metrics import accuracy_score, classification_report

predictions = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, predictions)}")
print(classification_report(y_test, predictions))

In [0]:
upcoming = spark.table("gold_dev.default.match_features").toPandas()

X_future = upcoming[["HomeTeamElo", "AwayTeamElo", "EloDiff", "IsNeutral"]]
X_future_scaled = scaler.transform(X_future)
predictions = model.predict_proba(X_future_scaled)[:,1]

upcoming["HomeWinProb"] = predictions.round(2)
upcoming["AwayWinProb"] = (1 - predictions).round(2)

display(upcoming)
